# 0 Introduction to SuctionNet

SuctionNet-1Billion is a benchmark built specifically to evaluate suction grasping in realistic, cluttered tabletop scenes. It reuses the raw data from GraspNet-1Billion and adds suction-specific supervision labels. In total there are 190 scenes, populated from a set of 88 everyday objects, with each one having its related 3D CAD model.

In this dataset, a "scene" means a particular arrangement of several objects (about 10 on average) placed on a tabletop in random poses and displacements. For every scene, two synchronized RGB-D cameras (Intel RealSense D435 and Azure Kinect) move along a fixed trajectory around the table, covering 256 distinct viewpoints.

Additionally, for each set of objects, there are typically four distinct scenes, where only the placements and orientations of those objects change, but the objects remain the same.

The goal of this notebook is to explore SuctionNet and, mainly:
1. Convert selected RGB-D images to 3D point clouds (using intrinsics + depth)
2) Extract per-object point clouds. This will enable another cross-domain testing (i.e. aligning a CAD model, source, to a real acquired point cloud, target), similarly to what we did with the company's acquired data (notebook `7-real_data_test.ipynb`.

This dataset provides data from 2 cameras: Kinect and RealSense. We will use the latter since it is the most common in industrial applications.

# 1 Setup

As always, we begin by importing the libraries we will use.

In [ ]:
import open3d as o3d
import copy
import numpy as np
from urllib.request import urlretrieve
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
import gdown
import zipfile
import os
from tqdm.auto import tqdm
from PIL import Image
from statistics import mode

# 2 Data Input

We can donwload the SuctionNet dataset from: https://graspnet.net/suction.

We could download the objects' 3D CAD models as we did for our acquired real data test. However, the author already makes available the [dense point clouds of these models](https://drive.google.com/file/d/1VuyNiJKwwt_lUlTk7BPT_uZsEI8o_Y2W/view?usp=sharing). These are basically the same (the dense point clouds are sampled from the models), so we'll directly adopt the **dense point clouds as our source**, allowing us to skip the step of converting the mesh (3D CAD model) to a point cloud (as detailed in section `2.1 Treating CAD Model (Source)` of notebook `7-real_data_test_.ipynb`).

As for our target, we will have the real acquired point clouds of the whole scene, with multiple objects, and the reduced point cloud of each single object. For this, we download the [train_4.zip](https://drive.google.com/file/d/1e8Xy7-lFhiXk0ugPOKvHKDiGTparmx00/view?usp=sharing) file, simply because it is the smaller part of the dataset, making it easier to handle. Note that there is no issue in using the so called "trainining" data because this was not used to train our deep-learning model, therefore there is no difference between using samples from this or from the actual "testing" split.

In [ ]:
# Guarantee the acquired data folder exists
data_dir = '../data/SuctionNet'
os.makedirs(data_dir, exist_ok=True)

# Download, if necessary, the files
models_zip_path = f'{data_dir}/models_dense_clouds.zip'
if not os.path.isfile(models_zip_path):
    gdown.download('https://drive.google.com/uc?id=1VuyNiJKwwt_lUlTk7BPT_uZsEI8o_Y2W', models_zip_path, quiet=False)
else:
    print(f'File {models_zip_path} already exists, skipping download.')

acquired_zip_path = f'{data_dir}/train_4.zip'
if not os.path.isfile(acquired_zip_path):
    gdown.download('https://drive.google.com/uc?id=1e8Xy7-lFhiXk0ugPOKvHKDiGTparmx00', acquired_zip_path, quiet=False)
else:
    print(f'File {acquired_zip_path} already exists, skipping download.')

# Extract the zips, if necessary
models_dir = f'{data_dir}/models'
if not os.path.exists(models_dir):
    print(f'Unzipping {models_zip_path}')
    with zipfile.ZipFile(models_zip_path, 'r') as zf:
            zf.extractall(models_dir)
    print(f'\tDone. Extracted to: {models_dir}')
else:
     print(f'Folder {models_dir} already exists, skipping extraction')

acquired_dir = f'{data_dir}/acquired/raw'
if not os.path.exists(acquired_dir):
    print(f'Unzipping {acquired_zip_path}')
    with zipfile.ZipFile(acquired_zip_path, 'r') as zf:
            zf.extractall(acquired_dir)
    print(f'\tDone. Extracted to: {acquired_dir}')
else:
     print(f'Folder {acquired_dir} already exists, skipping extraction')

print('Dataset ready!')

# 3 Converting Models' Dense Clouds

The first thing we need to do is convert the .npz files of the dense point clouds files to actual .ply files that can be handled by Open3D.

In [ ]:
def make_open3d_point_cloud(xyz, normals=None, color=None):
    "Create open3d point cloud object from array of points"
    pcd = o3d.geometry.PointCloud()                         # initializes PC object
    pcd.points = o3d.utility.Vector3dVector(xyz)            # extract points
    if normals is not None:
        pcd.normals = o3d.utility.Vector3dVector(normals)   # extract normals
    if color is not None:
        pcd.colors = o3d.utility.Vector3dVector(color)      # extract colors
    return pcd


def npz_to_ply(obj_id, models_dir):
    "Convert .npz (compressed numpy array) files to .ply files (point clouds)"

    # Read the npz data
    npz_path = f'{models_dir}/dense_point_clouds/{obj_id:03d}.npz'
    npz = np.load(npz_path)

    # Extract the points and normals
    xyz = npz['points']
    normals = npz.get('normals')

    # Convert it to pcd and save it as a .ply file
    pcd = make_open3d_point_cloud(xyz, normals)
    ply_dir = f'{models_dir}/ply'
    os.makedirs(ply_dir, exist_ok=True)                     # creates output dir if it doesn't exist yet
    ply_file = f'{ply_dir}/{obj_id:03d}.ply'                    # sets proper output file inside the output dir
    o3d.io.write_point_cloud(ply_file, pcd)                 # writes the .ply file
    return


def convert_npz_folder(models_dir):
    folder = f'{models_dir}/dense_point_clouds'
    for i in tqdm(range(len(os.listdir(folder))), 'Converting...'):
        npz_to_ply(i, models_dir)
    return

In [ ]:
convert_npz_folder(models_dir)

We can then try to visualize one of these to check whether it worked.

In [ ]:
test = o3d.io.read_point_cloud(f'{models_dir}/ply/003.ply')
o3d.visualization.draw_plotly([test], point_sample_factor=0.05, width=900, height=600)

# 4 Visualization of the Dataset

This section focuses on generating different visualizations of the dataset. For each scene (i.e., each arrangement and displacement of objects), images were captured from multiple angles. The code below extracts the images across these angles to visualize the path the robot arm took with the camera. It also compiles the images into a GIF.

In [ ]:
def get_gif_angle_variation(scene_id, data_dir):

    # Get rgb directory of that scene
    scene_rgb = f'{data_dir}/acquired/raw/scene_{scene_id:04d}/realsense/rgb'

    # Create output folder
    output_folder = f"{data_dir}/visualizations"
    os.makedirs(output_folder, exist_ok=True) 

    # Output GIF path
    out_path = f'{output_folder}/scene_{scene_id:04d}_angle_variation.gif'

    images = []

    # For each rgb image (sorted in increasing ID order)
    for rgb in sorted(os.listdir(scene_rgb)):
        rgb_path = f'{scene_rgb}/{rgb}'
        images.append(Image.open(rgb_path).convert("RGBA"))        # read it with Pillow

    # Save GIF (duration is ms per frame)
    images[0].save(
        out_path,
        save_all=True,
        append_images=images[1:],
        duration=int(10000 / 24),
    )

    print(f'GIF saved at: {out_path}')

    return

In [ ]:
# Select the desired scene's ID (here, it goes from 90 to 99)
get_gif_angle_variation(90, data_dir)

Another aspect we can visualize is how the set of objects, and their displacements, changes across scenes. Since all scenes include images from the same angles, we can fix one viewpoint and create a GIF that cycles through the different scenes at that position.

In [ ]:
def get_gif_scene_variation(view_id, data_dir):

    # Create output folder
    output_folder = f"{data_dir}/visualizations"
    os.makedirs(output_folder, exist_ok=True)

    # Output GIF path
    out_path = f'{output_folder}/view_{view_id:04d}_scene_variation.gif'

    images = []

    for scene in sorted(os.listdir(f'{data_dir}/acquired/raw')):

        # Get rgb directory of that scene
        view_rgb = f'{data_dir}/acquired/raw/{scene}/realsense/rgb/{view_id:04d}.png'

        # read it with Pillow
        images.append(Image.open(view_rgb).convert("RGBA"))        

    # Save GIF (duration is ms per frame)
    images[0].save(
        out_path,
        save_all=True,
        append_images=images[1:],
        duration=int(15000 / 24),
        disposal=2,
        loop=0
    )

    print(f'GIF saved at: {out_path}')

In [ ]:
# Select the desired view's ID (from 0 to 255)
get_gif_scene_variation(0, data_dir)

# 5 Converting Images to Point Clouds

Our goal is to convert 2D RGB-D images into 3D point clouds. The dataset provides, for each scene and each view angle, a color image (RGB) and a matched depth map where each pixel’s value encodes its distance from the camera rather than its color. By combining these, we can recover a 3D point in space for each pixel, yielding an RGB-D point cloud [4].

Additionally, to do this conversion, we also need the camera’s parameters. Below, there is a quick recap of what the camera’s parameters are and how we use them.

## 5.1 Theory

#### Intrinsics and Extrinsics

**Intrinsics** describe the camera’s internal geometry. These are the parameters of the pinhole model that map 3D points in the camera coordinate frame (i.e., the camera’s refence coordinate system with origin at the optical center) to the image plane. The main intrinsic parameters are included in the intrinsic matrix $K$.

$$
K=\begin{bmatrix} f_x & 0 & o_x\\ 0 & f_y & o_y\\ 0 & 0 & 1 \end{bmatrix}
$$
- $f_x$, $f_y$: focal lengths, in pixels  
- $o_x$, $o_y$: principal point's (optical center) coordinates in pixels.

Then, given a 3D point $\mathbf{p}_c=(x_c,y_c,z_c)^\top$ in the camera frame, we can project to image coordinates $(u,v)$ via:

$$
\begin{aligned}
u &= \frac{f_x\,x_c}{z_c} + o_x,\\
v &= \frac{f_y\,y_c}{z_c} + o_y,
\end{aligned}
\qquad\text{or}\qquad
\tilde{\mathbf{x}} \sim K\,[I\;|\;0]\;\tilde{\mathbf{p}}_c
$$

Note that besides the intrinsic parameters included in $K$, lens distortion parameters may be used for a full calibration if applicable [2].

**Extrinsics**, on the other hand, describe the pose of the camera in a larger coordinate system (e.g., the robot or world frame). They consist of a rotation $R \in SO(3)$ and a translation $\mathbf{t} \in \mathbb{R}^3$:

$$
\mathbf{p}_w \;=\; R\,\mathbf{p}_c \;+\; \mathbf{t}
\quad\text{or in homogeneous form}\quad
T = \begin{bmatrix} R & \mathbf{t} \\ \mathbf{0}^\top & 1 \end{bmatrix},\quad
\tilde{\mathbf{p}}_w = T \,\tilde{\mathbf{p}}_c
$$

Together, intrinsics and extrinsics form the standard camera matrix $P = K\,[R\;|\;t]$, which maps world points to image points $(\tilde{\mathbf{x}} \sim P\,\tilde{\mathbf{X}}_w)$ [2][5].

#### From pixels to 3D (back-projection)

Given a pixel $(u,v)$ with depth $Z_c$ (in meters), we invert the pinhole equations to recover its 3D position in the camera frame:

$$
\begin{aligned}
x_c &= \frac{(u - o_x)}{f_x}\,Z_c,\\
y_c &= \frac{(v - o_y)}{f_y}\,Z_c,\\
z_c &= Z_c,
\end{aligned}
\qquad\Longleftrightarrow\qquad
\mathbf{p}_c = Z_c\,K^{-1}\!\begin{bmatrix}u\\v\\1\end{bmatrix}
$$

To obtain the points in a world/robot frame, we apply the inverse extrinsics: $\mathbf{p}_c = R^\top\,(\mathbf{p}_w - \mathbf{t})$ [1][2][3]. Additioanlly, RGB values from the color image at $(u,v)$ can be attached to each 3D point to form a colored point cloud, if desired [4].

**Depth units & scale:** we must ensure the depth is converted to meters. Many RGB-D devices (e.g., Intel RealSense) store integer depth units that require a device-specific scale factor to convert to meters [6][7].

In our particular case, since we are not interested in reconstructing the absolute position of the 3D points, but rather the object itself, we don’t need extrinsics, only the intrinsics. To this end, the authors made available the intrinsic matrix of the camera in the file `camK.npy`, present in each scene.

#### References

[1] OpenCV, _Camera Calibration and 3D Reconstruction_ (pinhole projection formulas and notation).  
[2] Hartley & Zisserman, _Multiple View Geometry in Computer Vision_, Cambridge Univ. Press (camera matrix, intrinsics/extrinsics).  
[3] Stanford CS231A notes, _Camera Models_ (pinhole model).  
[4] Open3D Docs, _RGB-D images -> point cloud_ (practical conversion APIs).  
[5] CMU 16-385 slides, _Camera Matrix_.  
[6] OpenCV, _Camera Calibration (Python tutorial)_ (intrinsics, extrinsics, distortion; undistortion workflow).  
[7] Intel RealSense Dev Docs, _Projection in SDK 2.0_ (coordinate systems, projection/back-projection with RealSense).

## 5.2 Demonstration

We begin by presenting how to do this conversion of one specific scene. Later we will expand the concepts presented here in a loop to perform it on more cases.

### 5.2.1 Complete Cloud

The first demonstration focus on how to convert the whole 2D picture to a point cloud, capturing the information of the whole scene and its objects.

The first step is to load the intrinsic matrix from the selected camera and scene.

In [ ]:
scene_id = 90

# Get scene directory
scene_dir = f'{acquired_dir}/scene_{scene_id:04d}'

# Load camera intrinsic matrix
camK = np.load(f'{scene_dir}/realsense/camK.npy')
print(camK)

Now we will use the `PinholeCameraIntrinsic` class from `Open3D` which stores the camera's intrinsic matrix, and image height and width ([reference](https://www.open3d.org/docs/latest/python_api/open3d.camera.PinholeCameraIntrinsic.html)).

In [ ]:
# Create Open3D camera intrinsic object and stores our matrix
intrinsic = o3d.camera.PinholeCameraIntrinsic()
intrinsic.intrinsic_matrix = camK

Another things we must select is the viewing angle. Thus, we define the ID of the image within the selected scene (i.e. the ID of where the camera was in the trajectory) and we use it to refer to both the color and depth images, guaranteeing that we are taking corresponding images.

In [ ]:
image_id = 1

# Gets directories for RGB, label and depth images
rgb_dir = f'{scene_dir}/realsense/rgb'
depth_dir = f'{scene_dir}/realsense/depth'
label_dir = f'{scene_dir}/realsense/label'

# Load images
rgb_img = o3d.io.read_image(f'{rgb_dir}/{image_id:04d}.png')
depth_img = o3d.io.read_image(f'{depth_dir}/{image_id:04d}.png')
label_img = o3d.io.read_image(f'{label_dir}/{image_id:04d}.png')

# Conver them to numpy arrays for easier manipulation
rgb = np.array(rgb_img)
depth = np.array(depth_img)
label = np.array(label_img)

We can then visualize the sample we selected.

In [ ]:
def plot_sample_images(rgb, depth, label=None, save_path=None, show=True):
    # Create a figure with subplots side by side
    fig_cols = 2 if label is None else 3
    fig, axes = plt.subplots(1, fig_cols, figsize=(15, 5))

    # Plot the RGB image on the left
    axes[0].imshow(rgb)
    axes[0].set_title('RGB', fontsize=16)
    axes[0].axis('off')  # Hide axes

    # Plot the depth image on the middle with a grayscale colormap
    axes[1].imshow(depth, cmap='gray')
    axes[1].set_title('DEPTH', fontsize=16)
    axes[1].axis('off')

    if label is not None:
        # Plot the label image on the right
        axes[2].imshow(label, cmap='gray')
        axes[2].set_title('LABEL', fontsize=16)
        axes[2].axis('off')

    # Display and save the plot
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path)
    if show:
        plt.show()
    plt.close(fig)

In [ ]:
plot_sample_images(rgb, depth)

Once we have the intrinsics and the images, we can convert the depth images into a point cloud. The first step is to merge both RBG and depth images into one. To do this, we use the `RGBDImage` class from Open3D, which stores both images into a single object, and has a built-in method for converting it to a point cloud. ([reference](https://www.open3d.org/docs/latest/python_api/open3d.geometry.RGBDImage.html))

In [ ]:
def image_to_point_cloud(rgb_img, depth_img, intrinsic):
    "Creates a PointCloud object from a sample's rgb and depth images"

    # Convert depth and RGB images into Open3D formats (class to store both images into a single object)
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(rgb_img,
                                                                    depth_img,
                                                                    depth_scale=1000.0,  # realsense, depth in mm
                                                                    convert_rgb_to_intensity=False)

    # Create a point cloud from the RGBD image and camera intrinsics
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_image, intrinsic)

    # Optionally, flip the point cloud to align with Open3D coordinate system
    # otherwise, we see upside down
    # in Open3D, positive is right, up and out of screen
    # for RealSense, positive is right, down and into the screen
    pcd.transform([[1, 0, 0, 0],
                   [0, -1, 0, 0],
                   [0, 0, -1, 0],
                   [0, 0, 0, 1]])

    return pcd

In [ ]:
pcd = image_to_point_cloud(rgb_img, depth_img, intrinsic)

o3d.visualization.draw_plotly([pcd], point_sample_factor=0.1, width=900, height=600)

### 5.2.2 Single Object

After obtaining the cloud of the whole scene, we need to extract the clouds of individual objects in a scene. To do this, we leverage the label image, which the author provides, along the RGB and Depth images, for each scene and each view.

The label is another 2D image, but here the value of each pixel (i.e., its brightness) is an integer that corresponds to the ID of the object appearing on that pixel or 0 if there is no object there (i.e., the background).

We can observe this in the following plot, where we insert the values on top of the pixels, to show the object's ID where they are located in the image.

In [ ]:
def pad_to_multiple(label, block_size):
    # Get original dimensions of the image
    cur_height, cur_width = label.shape

    # Get how much padding is needed to reach the desired block's height and width
    # Using the negative here, returns us how much to add to cur_# to reach block_# 
    pad_h = (-cur_height) % block_size
    pad_w = (-cur_width) % block_size

    # If the current size is already compatible (i.e. equal or multiple) return as it is
    if pad_h == 0 and pad_w == 0:
        return label
    
    # Pad on the righ and bottom of the image with values 0 (i.e. the background)
    padded = np.pad(label, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)
    return padded


def majority_pool(label, block_size):

    # Pad so the shape is divisible by block size (so we can divide the image in blocks)
    padded = pad_to_multiple(label, block_size)
    height, width = padded.shape

    # Count how many integers groups we can fit vertically and horizontally (not equal because image is not square)
    num_blocks_vert = height // block_size
    num_blocks_horiz = width // block_size
    
    # Dive the image in blocks (i.e. each block contains a portion of the pixels)
    blocks = padded.reshape(num_blocks_vert, block_size, num_blocks_horiz, block_size)

    # Now, image's axes are not (y, x), but: (block_row, within_block_row, block_col, within_block_col)
    # So we reorder the axes so the two block indices are next to each other: (block_row, block_col, within_block_row, within_block_col)
    blocks = blocks.transpose(0, 2, 1, 3)

    # Reshape to flatten the axes inside each block (i.e. instead of being a matrix, it becomes an 1D vector)
    # So our shape becomes: (block_row, block_col, indice)
    blocks = blocks.reshape(height//block_size, width//block_size, block_size**2)

    # Our output will be one pixel per group, so the size is given by the amount of groups
    out = np.empty((num_blocks_vert, num_blocks_horiz))

    # For each block, get the maximum value (maximum pooling), as a deterministic way
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = mode(blocks[i, j, :])
    return out


def plot_label_with_IDs(label, block_size, rgb, save_path=None, show=True):

    # Downsample label image for one pixel per group (max pool)
    label_down = majority_pool(label, block_size)
    height, width = label_down.shape

    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(15,10), dpi=175)
    ax[0].imshow(label_down, interpolation='nearest', cmap='Paired')
    ax[0].set_title("")
    ax[0].axis('off')

    # Add text at cell centers (no scaling needed now)
    for y in range(height):
        for x in range(width):
            v = int(label_down[y, x])
            ax[0].text(x, y, str(v) if v != 0 else "",
                    ha='center', va='center', fontsize=int(0.3*block_size), color='white')

    # Grid at cell boundaries (now each cell is 1x1 pixel)
    ax[0].set_xticks(np.arange(-0.5, width, 1))
    ax[0].set_yticks(np.arange(-0.5, height, 1))
    ax[0].grid(which='both', linestyle='-', linewidth=0.5, color='gray')
    ax[0].set_xticklabels([])
    ax[0].set_yticklabels([])

    # RGB image for reference
    ax[1].imshow(rgb)
    ax[1].set_yticks([])
    ax[1].set_xticks([])

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path)
    if show:
        plt.show()
    plt.close(fig)


plot_label_with_IDs(label, block_size=16, rgb=rgb)

We can use the label images to create masks that isolate a specific object. Given the desired object’s ID, we compare each pixel in the label image against this ID: if the values match, the pixel belongs to the object and is set to 1; otherwise, it is set to 0. This produces a binary mask in which the selected object’s region is 1, and all other areas are 0, effectively filtering out everything else.

Once this mask is generated, we apply it to both the RGB and depth images to extract only the relevant pixels. From these filtered images, we follow the same approach as before to generate a point cloud containing just the selected object rather than the entire scene.

We begin by checking the objects present in the current scene.

In [ ]:
obj_list = f'{scene_dir}/object_id_list.txt'
with open(obj_list, 'r') as file:
    file_content = file.read() # Read the content of the file
    print('Object IDs List:\n%s' %file_content)

In [ ]:
print("Unique labels in the label image:", np.unique(label))

There's a difference of 1 unit here because the object list is 0-indexed while the labels are 1-indexed (in this case, 0 is reserved for the background).

Now, we must select which object we want. The full list of objects and their IDs can be found [here](https://graspnet.net/images/obj_list.pdf).

Then, we can look for pixels matching this value as create our mask.

In [ ]:
object_ID = 5

# Create a mask that selects only the pixels corresponding to the object
label_ID = object_ID + 1
mask = (label == label_ID)
plt.imshow(mask)
plt.axis('off')
plt.show()

Now we can apply it to the RGB and Depth images.

In [ ]:
# Apply the mask to the depth image by setting non-object pixels to zero
masked_depth = depth                                # Convert depth image no np array
masked_depth[~mask] = 0                             # Set background and other objects' depth to zero

# Apply the mask to the RGB
masked_rgb = rgb
masked_rgb[~mask] = 0

plot_sample_images(masked_rgb, masked_depth, mask)

And, finally, we obtain our reduced point cloud.

In [ ]:
# Convert the masked images back to an Open3D depth image
masked_depth = o3d.geometry.Image(masked_depth)
masked_rgb = o3d.geometry.Image(masked_rgb)

obj_pcd = image_to_point_cloud(masked_rgb, masked_depth, intrinsic)

o3d.visualization.draw_plotly([obj_pcd], point_sample_factor=0.5, width=900, height=600)

## 5.3 Extraction Pipeline

Now that we have properly extracted both the complete and reduced point cloud, we define a pipeline to extract these from every sample we have available.

The obtained dataset will be stored in the following way:

```txt
data
|  SuctionNet
|  |  model
|  |  acquired
|  |  |  raw
|  |  |  |  {scene}
|  |  |  |  |  {image}
|  |  |  full_scene
|  |  |  |  {scene}
|  |  |  |  |  {image}
|  |  |  single_object
|  |  |  |  {scene}
|  |  |  |  | {image}
|  |  |  |  |  |  {object}
```

In [ ]:
def run_cloud_extractor(acquired_dir):

    # List scenes up-front so tqdm has a total (ignore non-folders)
    scenes = [s for s in os.listdir(acquired_dir) if os.path.isdir(os.path.join(acquired_dir, s))]

    # For each scene in the given directory
    for scene in tqdm(scenes, desc='Processing scenes', unit='scene', position=0):
        scene_dir = f'{acquired_dir}/{scene}'
        
        # Load camera intrinsic matrix
        camK = np.load(f'{scene_dir}/realsense/camK.npy')
        
        # Create Open3D camera intrinsic object to store the matrix
        intrinsic = o3d.camera.PinholeCameraIntrinsic()
        intrinsic.intrinsic_matrix = camK

        # Gets directories for RGB, label and depth images
        rgb_dir = f'{scene_dir}/realsense/rgb'
        depth_dir = f'{scene_dir}/realsense/depth'
        label_dir = f'{scene_dir}/realsense/label'

        # Create output folders for that scene
        full_dir = f'{data_dir}/acquired/full_scene/{scene}'
        os.makedirs(full_dir, exist_ok=True)
        single_dir = f'{data_dir}/acquired/single_object/{scene}'
        os.makedirs(single_dir, exist_ok=True)

        # Pre-list image files for totals & stable order
        images = sorted([f for f in os.listdir(rgb_dir) if f.endswith('.png')])
        pbar_imgs = tqdm(images, desc=f"{scene}: images", unit="img", leave=False, position=1)

        for image_id in pbar_imgs:

            # Extract numeric ID from filename
            image_id = int(image_id.split('.')[0])
            
            # Load images
            rgb_img = o3d.io.read_image(f'{rgb_dir}/{image_id:04d}.png')
            depth_img = o3d.io.read_image(f'{depth_dir}/{image_id:04d}.png')
            label_img = o3d.io.read_image(f'{label_dir}/{image_id:04d}.png')

            # Conver them to numpy arrays for easier manipulation
            rgb = np.array(rgb_img)
            depth = np.array(depth_img)
            label = np.array(label_img)
        
            # Create full scene output folder for the current image
            image_full_dir = f'{full_dir}/image_{image_id:04d}'
            os.makedirs(image_full_dir, exist_ok=True)
            
            # Save visualization images (all samples and label with IDs)
            plot_sample_images(rgb, depth, label, save_path=f'{image_full_dir}/all_samples_{scene}_image_{image_id:04d}.jpg', show=False)
            plot_label_with_IDs(label, block_size=16, rgb=rgb, save_path=f'{image_full_dir}/label_with_IDs_{scene}_image_{image_id:04d}.jpg', show=False)

            # Extract and save full point cloud
            pcd = image_to_point_cloud(rgb_img, depth_img, intrinsic)
            o3d.io.write_point_cloud(f'{image_full_dir}/pcd.ply', pcd)

            # Get the list of object IDs in the scene (ignoring the background, ID 0)
            obj_list = np.unique(label)
            obj_list = obj_list[obj_list != 0]

            # Update tqdm with current image and number of objects found
            pbar_imgs.set_postfix(img=f"{image_id:04d}", objs=len(obj_list))

            # For each object in the current image of the current scene
            for obj_id in tqdm(obj_list, desc=f"{scene} img {image_id:04d}: objects", unit="obj", leave=False, position=2):

                # Create output folder for that object
                obj_dir = f'{single_dir}/image_{image_id:04d}/object_{obj_id:02d}'
                os.makedirs(obj_dir, exist_ok=True)

                # Create a mask that selects only the pixels corresponding to the object
                mask = (label == obj_id)

                # Extract the RGB and depth images for the current object
                depth_obj = copy.deepcopy(depth)
                rgb_obj = copy.deepcopy(rgb)
                depth_obj[~mask] = 0
                rgb_obj[~mask] = 0

                # Save the object-specific images
                plot_sample_images(rgb_obj, depth_obj, mask, save_path=f'{obj_dir}/object_samples_{scene}_image_{image_id:04d}_object_{obj_id:02d}.jpg', show=False)

                # Convert the masked images back to an Open3D depth image
                depth_obj = o3d.geometry.Image(depth_obj)
                rgb_obj = o3d.geometry.Image(rgb_obj)

                # Extract and save the point cloud for the current object
                obj_pcd = image_to_point_cloud(rgb_obj, depth_obj, intrinsic)
                o3d.io.write_point_cloud(f'{obj_dir}/pcd.ply', obj_pcd)

        # Let the image bar disappear when moving to the next scene
        pbar_imgs.close()

In [ ]:
run_cloud_extractor(acquired_dir)

With the extracted point clouds, we can move into our final test using this data, which will be done in the next notebook.